# Class 1
# Convolutional Neural Networks (CNN)

This notebook introduces:

1. **Convultions**
2. **Datasets**
3. **Custom CNN**: A small convolutional network and how an image is transformed layer by layer.
4. **ResNet50**: A modern architecture with skip connections and how it works.

We use **PyTorch** for implementation and **matplotlib** for visualization.


In [ ]:
! git clone https://github.com/EzequielMatiasArevalo/tuia-computer-vision.git || ( cd tuia-computer-vision && git pull origin main)
! mkdir -p media && ( mv tuia-computer-vision/media/pictures media/ || cp -r tuia-computer-vision/media/pictures/* media/pictures/ )
! pip install -r tuia-computer-vision/requirements.txt

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
from PIL import Image
from scipy import signal
from cv2.typing import MatLike
from typing import Optional
from numpy.typing import NDArray
import requests
DEFAULT_OUTPUT_PATH = "./media/outputs/class1"
DEFAULT_SAMPLE_IMG_PATH="./media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/sample.jpeg"

# 1- Convulutions

## 1.1 : Creating Dataset
We will download a random image from the COCO dataset, use it as an example, and save it to the local filesystem.

In [ ]:

image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")
image.save(DEFAULT_SAMPLE_IMG_PATH)
plt.figure(figsize=(10, 7))
plt.imshow(image)
plt.title("Input Image")
plt.axis("off")
plt.show()

## 1.2 : Display images using matplot
We use PIL, OpenCV, and Matplotlib to load the image saved in the previous step, demonstrating the same result across different libraries.

In [ ]:
def load_and_display_images(img_path : str, output_path : str = None)-> MatLike:
    """
    Demonstrate different ways to load images with various libraries
    """
    if output_path is None:
        output_path = f"{DEFAULT_OUTPUT_PATH}/loaded_images_comparison.png"

    if not os.path.exists(os.path.dirname(output_path)):
        os.makedirs(os.path.dirname(output_path))

    # Method 1: Using PIL
    img_pil = Image.open(img_path)
    print(f"PIL Image - Size: {img_pil.size}, Mode: {img_pil.mode}")

    # Method 2: Using OpenCV (BGR format)
    img_cv2 = cv2.imread(img_path)
    img_cv2_rgb = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)
    print(f"OpenCV Image - Shape: {img_cv2.shape}, Dtype: {img_cv2.dtype}")

    # Method 3: Using matplotlib
    img_plt = plt.imread(img_path)
    print(f"Matplotlib Image - Shape: {img_plt.shape}, Dtype: {img_plt.dtype}")

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_pil)
    axes[0].set_title('PIL Image')
    axes[0].axis('off')

    axes[1].imshow(img_cv2_rgb)
    axes[1].set_title('OpenCV Image')
    axes[1].axis('off')

    axes[2].imshow(img_plt)
    axes[2].set_title('Matplotlib Image')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
    plt.savefig(output_path)
    plt.close()

    return img_cv2_rgb

In [ ]:
print("Lab 1: Image Processing Fundamentals")
print("=" * 50)

print("\n1. Loading and displaying images...")
lab_img = load_and_display_images(img_path=DEFAULT_SAMPLE_IMG_PATH)

## 1.3 : Color Space Conversions
We rearrange the channels, convert the image from RGB to HSV, and display each channel to illustrate different ways of representing an image.

In [ ]:
from collections import namedtuple
from typing import NamedTuple


def color_space_exploration(img : MatLike, output_path : Optional[str] = None)->NamedTuple:
    """
    Explore different color spaces and their properties
    """
    if output_path is None:
        output_path = f"{DEFAULT_OUTPUT_PATH}/color_spaces.png"

    if not os.path.exists(os.path.dirname(output_path)):
        os.makedirs(os.path.dirname(output_path))

    # Load image
    img_rgb = img.copy()

    # Convert to different color spaces
    img_gray = cv2.cvtColor(img.copy(), cv2.COLOR_RGB2GRAY)
    img_hsv = cv2.cvtColor(img.copy(), cv2.COLOR_RGB2HSV)
    img_lab = cv2.cvtColor(img.copy(), cv2.COLOR_BGR2LAB)

    # Visualize color spaces
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # RGB
    axes[0, 0].imshow(img_rgb)
    axes[0, 0].set_title('RGB Image')
    axes[0, 0].axis('off')

    # Grayscale
    axes[0, 1].imshow(img_gray, cmap='gray')
    axes[0, 1].set_title('Grayscale')
    axes[0, 1].axis('off')

    # RGB Channels
    channels = ['Red', 'Green', 'Blue']
    for i, (channel, color) in enumerate(zip(cv2.split(img_rgb), ['Reds', 'Greens', 'Blues'])):
        axes[0, 2].imshow(channel, cmap=color)
        axes[0, 2].set_title('RGB Channels (overlayed)')
        axes[0, 2].axis('off')

    # HSV Channels
    h, s, v = cv2.split(img_hsv)
    axes[1, 0].imshow(h, cmap='hsv')
    axes[1, 0].set_title('HSV - Hue')
    axes[1, 0].axis('off')

    axes[1, 1].imshow(s, cmap='gray')
    axes[1, 1].set_title('HSV - Saturation')
    axes[1, 1].axis('off')

    axes[1, 2].imshow(v, cmap='gray')
    axes[1, 2].set_title('HSV - Value')
    axes[1, 2].axis('off')

    plt.tight_layout()
    plt.show()
    plt.savefig(output_path)
    plt.close()

    # Color space conversion formulas
    print("\nColor Space Conversion Insights:")
    print(f"RGB range: [0, 255]")
    print(f"HSV - H range: [0, 180], S range: [0, 255], V range: [0, 255]")
    print(f"Grayscale formula: 0.299*R + 0.587*G + 0.114*B")

    result = namedtuple('Images', ['rgb','hsv', 'gray'])
    return result(img_rgb, img_hsv, img_gray)

In [ ]:
print("\n2. Color space exploration...")
img_rgb, img_hsv, img_gray = color_space_exploration(lab_img)

##1.4 How convultions works
Convolution is an operation used in image processing to extract features (edges, textures, patterns) by applying a small matrix called a **kernel** (or filter) over an image.

### How it works

1. **Kernel definition**
   A kernel is a small matrix (e.g., 3×3) with predefined values. Each kernel is designed to highlight specific features (e.g., edges, blur).

2. **Sliding window**
   The kernel is placed over a region of the image and slides across it pixel by pixel (left to right, top to bottom).

3. **Element-wise multiplication**
   For each position, the kernel values are multiplied element-wise with the corresponding image pixels.

4. **Summation**
   The results of these multiplications are summed to produce a single output value.

5. **Output pixel**
   This value becomes the new pixel value in the output (filtered) image at that position.

### Intuition

* The kernel acts like a **feature detector**.
* It emphasizes certain patterns depending on its values:

  * **Sobel** → detects edges
  * **Gaussian blur** → smooths the image
  * **Identity** → keeps the image unchanged

### Example (3×3 kernel)

For a small image patch:

```
Image patch:        Kernel:
[ 1  2  3 ]         [ 0  1  0 ]
[ 4  5  6 ]   *     [ 1 -4  1 ]
[ 7  8  9 ]         [ 0  1  0 ]
```

Compute:

```
(1·0 + 2·1 + 3·0 +
 4·1 + 5·(-4) + 6·1 +
 7·0 + 8·1 + 9·0) = result
```

This result becomes one pixel in the output image.

### Key concepts

* **Stride**: how many pixels the kernel moves each step
* **Padding**: adding borders to control output size
* **Channels**: for RGB images, convolution is applied per channel and then combined

### Why it matters

Convolution is the core operation behind:

* Image filtering (OpenCV, PIL, etc.)
* Feature extraction
* Deep learning models like CNNs (Convolutional Neural Networks)

It transforms raw pixel data into meaningful patterns that algorithms can use.


In [ ]:
from IPython.display import Image, display

display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/convolved.gif'))
display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/convolved-2.gif',width=300,height=300))


### 1.4.1 : Convolution and Filtering
We will generate custom kernels to convolve images. As examples, we will create 3×3 kernels simulating Sobel, identity, Gaussian blur, and others.

In [ ]:
def apply_convolution_filters(img_gray : MatLike, output_path : Optional[str] = None)-> dict:
    """
    Demonstrate various convolution filters and their effects
    """
    if output_path is None:
        output_path = f"{DEFAULT_OUTPUT_PATH}/convolution_filters.png"
    if not os.path.exists(os.path.dirname(output_path)):
        os.makedirs(os.path.dirname(output_path))

    # Load grayscale image
    img = img_gray
    img = img.astype(np.float32)

    # Define various kernels
    kernels = {

        'Identity': np.array(
            [
                [0, 0, 0],
                [0, 1, 0],
                [0, 0, 0]
            ]
        ),

        'Gaussian Blur': np.array(
            [
                [1, 2, 1],
                [2, 4, 2],
                [1, 2, 1]
            ]
        ),

        'Box Blur': (0.5) * np.array(
            np.ones((5, 5))
            ),

        'Sharpen': np.array(
            [
                [0, -1, 0],
                [-1, 5, -1],
                [0, -1, 0]
            ]
        ),

        'Edge Detection (Sobel X)': np.array(
            [
                [-1, 0, 1],
                [-2, 0, 2],
                [-1, 0, 1]
            ]
        ),

        'Edge Detection (Sobel Y)': np.array(
            [
                [-1, -2, -1],
                [0, 0, 0],
                [1, 2, 1]
            ]
        ),

        'Edge Detection (Laplacian)': np.array(
            [
                [0, 1, 0],
                [1, -4, 1],
                [0, 1, 0]
            ]
        ),

        'Emboss': np.array(
            [
                [-2, -1, 0],
                [-1, 1, 1],
                [0, 1, 2]
            ]
        )
    }
   # Apply filters
    n_kernels = len(kernels)
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    axes = axes.flatten()

    # Original image
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    # Apply each kernel
    for idx, (name, kernel) in enumerate(kernels.items(), start=1):
        # Method 1: Using scipy
        filtered = signal.convolve2d(img, kernel, mode='same', boundary='symm')

        axes[idx].imshow(filtered, cmap='gray')
        axes[idx].set_title(f'{name}\nKernel shape: {kernel.shape}')
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()
    plt.savefig(output_path)
    plt.close()

    return kernels

In [ ]:
print("\n3. Applying convolution filters...")
kernels = apply_convolution_filters(img_gray)

## **1.5 :** Custom Convulutional 2D
We define a custom convolution function to apply our kernels to the image.

In [ ]:
def custom_convolution_2d(image: MatLike, kernel: MatLike, intensity: float = 0.5,padding: int =0, stride: int =1)->NDArray:
    """
    Implement 2D convolution from scratch

    Args:
        image: Input image (H x W)
        kernel: Convolution kernel (K x K)
        intensity: Intensity factor for the convolution result
        padding: Padding size
        stride: Stride size

    Returns:
        Convolved image
    """
    # Get dimensions
    kernel = kernel * intensity
    image_h, image_w = image.shape
    kernel_h, kernel_w = kernel.shape

    # Add padding
    if padding > 0:
        image = np.pad(image, padding, mode='constant')

    # Calculate output dimensions
    out_h = (image_h + 2 * padding - kernel_h) // stride + 1
    out_w = (image_w + 2 * padding - kernel_w) // stride + 1

    # Initialize output
    output = np.zeros((out_h, out_w))

    # Perform convolution
    for i in range(out_h):
        for j in range(out_w):
            # Extract region
            h_start = i * stride
            h_end = h_start + kernel_h
            w_start = j * stride
            w_end = w_start + kernel_w

            region = image[h_start:h_end, w_start:w_end]

            # Element-wise multiplication and sum
            output[i, j] = np.sum(region * kernel)

    return output

In [ ]:
print("\n4. Custom convolution implementation...")
custom_conv_img = custom_convolution_2d(image=img_gray, kernel=kernels["Identity"])
plt.imshow(custom_conv_img, cmap='gray')
plt.savefig(f"{DEFAULT_OUTPUT_PATH}/custom_convolution_gaussian_blur.png")
#plt.close()

# 2 - Datasets
Computer vision is a field of artificial intelligence that enables machines to interpret and understand the visual world through images and videos, emulating the capabilities of the human visual system. At the core of this field are datasets—structured collections of data that are essential for the training, evaluation, and development of computer vision algorithms.



### 2.1 Datasets role in computer vision:

- Model Training: Deep learning models, particularly Convolutional Neural Networks (CNNs), require large volumes of labeled data to learn how to recognize patterns and features in images. Without suitable datasets, the development of accurate and robust models would be impossible.

- Evaluation and Benchmarking: Standard datasets enable the objective evaluation of different algorithms and models. They provide a common framework for comparing results and establishing benchmarks that drive progress in the field.

- Diversity and Generalization: Diverse datasets that cover a wide range of scenarios, objects, and conditions are essential to ensure that developed models can generalize well to new situations rather than being limited to the specific data on which they were trained.

### Types of Datasets

There are several types of datasets used in computer vision, each suited for different tasks and applications. The most common include:

- Classification Datasets: Used to train models that assign labels to entire images. Notable examples include ImageNet and CIFAR-10.

- Object Detection Datasets: Designed to train models that identify and localize objects within images. Examples include MS COCO and PASCAL VOC.

- Segmentation Datasets: These datasets contain detailed annotations that allow models to delineate and segment objects within an image. MS COCO is also used for this purpose, along with Cityscapes for urban scene segmentation.

- Multiple Object Tracking Datasets: These datasets are essential for developing and evaluating algorithms capable of tracking multiple moving objects across video sequences. Notable examples include MOTChallenge, which provides various urban surveillance and public environment scenarios for pedestrian tracking; UA-DETRAC, specifically designed for vehicle tracking under diverse traffic conditions; and KITTI Tracking, which provides data captured from a moving vehicle, focusing on tracking cars, pedestrians, and cyclists in urban environments.

- Specialized Datasets: Some datasets are designed for very specific tasks, such as face detection (Labeled Faces in the Wild) or scene classification (MIT Places).


### 2.2 Challenges and Considerations

Despite their importance, the use of datasets in computer vision presents several challenges:

- Data Quality: The accuracy of models depends heavily on the quality of the training data. Poorly labeled or noisy data can lead to inaccurate models.

- Size and Accessibility: Large datasets require significant storage and processing resources. Additionally, not all datasets are publicly available, which can limit research and development.

- Ethics and Privacy: The use of images containing personal information raises privacy and ethical concerns. It is important to ensure that datasets comply with relevant regulations and ethical standards.

- Dataset Bias: Many datasets may contain inherent biases due to how the data is collected or annotated. These biases can lead to models that perpetuate or amplify discrimination, resulting in unfair or inaccurate applications. Identifying and mitigating these biases is essential for developing equitable and reliable models.

## 2.3 Annotation and labeling types :


### 2.3.1 Bounding Boxes

Bounding boxes are the most commonly used type of annotation in computer vision. They are rectangular boxes that define the location of a target object within an image. A bounding box is typically specified using the x and y coordinates of the top-left corner and the x and y coordinates of the bottom-right corner of the rectangle. Bounding boxes are generally used in object detection and object localization tasks.

In [ ]:

display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/bbox.png', width=300,height=300))
display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/bbox-2.png', width=300,height=300))
display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/bbox-yolo.png', width=300,height=300))
display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/bbox-3.png', width=300,height=300))



### 2.3.2 Polygonal Segmentation

Objects do not always have rectangular shapes. For this reason, polygonal segmentation is another type of data annotation in which complex polygons are used instead of rectangles to define the shape and location of an object with much greater precision.

In [ ]:

display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/seg-1.png', width=300,height=300))
display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/seg-2.png', width=300,height=300))

### 2.3.3 Semantic Segmentation

Semantic segmentation is a pixel-level annotation technique in which every pixel in an image is assigned to a specific class. These classes may include pedestrian, car, bus, road, sidewalk, and others, meaning that each pixel carries a semantic label.

Semantic segmentation is mainly used in scenarios where environmental context is very important. For example, it is widely used in autonomous vehicles and robotics, where models need to understand the environment in which they operate.

In [ ]:

display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/sem.png'))

### 2.3.4 Cuboides 3D
3D cuboids are similar to bounding boxes but include additional depth information about the object. With 3D cuboids, it is possible to obtain a three-dimensional representation of the object, allowing systems to distinguish characteristics such as volume and position in 3D space.

A common use case for 3D cuboids is in autonomous vehicles, where depth information can be used to measure the distance between objects and the vehicle.

In [ ]:
display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/cuboid.png'))

### 2.3.5 Keypoints and Landmarks

Keypoint and landmark annotation is used to detect small objects, structures, and shape variations by placing specific points on an image. This type of annotation is useful for identifying facial features, facial expressions, human body parts, and body poses.

In [ ]:

display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/pose.png'))

### 2.3.6 Lines and Splines

As the name suggests, this type of annotation is created using lines and splines. Splines are mathematical functions used to generate smooth curves that pass through a set of control points. This type of annotation is commonly used in autonomous vehicles for lane detection and recognition.

In [ ]:
display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/lines.png'))


## 2.4 Dataset Example


### 2.4.1 Download coco

In [ ]:
import os
import urllib.request
import zipfile
from pathlib import Path
from tqdm import tqdm  # pip install tqdm

Path("data").mkdir(exist_ok=True)
ROOT = Path("data/coco")
ROOT.mkdir(exist_ok=True)

DOWNLOAD_TRAIN = False
DOWNLOAD_VAL   = True
DOWNLOAD_TEST  = False

URLS = {
    # Images
    "train2017":      "http://images.cocodataset.org/zips/train2017.zip",        # ~18 GB
    "val2017":        "http://images.cocodataset.org/zips/val2017.zip",           # ~1  GB
    "test2017":       "http://images.cocodataset.org/zips/test2017.zip",          # ~6  GB (optional)

    # Annotations (detection + keypoints + captions + stuff + panoptic)
    "annotations":    "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",  # ~241 MB
    "test_info":      "http://images.cocodataset.org/annotations/image_info_test2017.zip",       # ~1  MB
}

class ProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download(url: str, dest: Path):

    print(f"  ↓ Downloading {dest.name} ...")
    with ProgressBar(unit="B", unit_scale=True, miniters=1, desc=dest.name) as t:
        urllib.request.urlretrieve(url, dest, reporthook=t.update_to)

def extract(zip_path: Path, dest: Path):
    print(f"  ⎋ Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest)
    zip_path.unlink()   # delete zip after extraction to save space
    print(f"  ✓ Done → {dest}")
# large, only needed for test-dev submissions

splits = {
    "train2017":   DOWNLOAD_TRAIN,
    "val2017":     DOWNLOAD_VAL,
    "test2017":    DOWNLOAD_TEST,
    "annotations": True,
    "test_info":   DOWNLOAD_TEST,
}

for name, enabled in splits.items():
    if not enabled:
        continue
    url      = URLS[name]
    zip_file = ROOT / Path(url).name

    if zip_file.exists():
        print(f"  ✓ Already exists: {zip_file}")
    else:
        print(f"  x Does not exist: {zip_file} ...")
        download(url, zip_file)
        extract(zip_file, ROOT)

print("\n✅ COCO download complete!")

### 2.4.2 Visualize coco examples

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision.datasets import CocoDetection
from torchvision import transforms

ROOT    = "data/coco/val2017"
ANN     = "data/coco/annotations/instances_val2017.json"

transform = transforms.ToTensor()

dataset = CocoDetection(root=ROOT, annFile=ANN, transform=transform)
coco    = dataset.coco

# Category id → name
id2label = {cat["id"]: cat["name"] for cat in coco.loadCats(coco.getCatIds())}

# ── Color palette (one per category) ─────────────────────────────────────────
np.random.seed(42)
COLORS = {cat_id: np.random.rand(3,) for cat_id in id2label}

# ── Visualize N samples ───────────────────────────────────────────────────────
def visualize_coco(dataset, num_images=6, cols=3):
    rows = (num_images + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
    axes = axes.flatten()

    for ax, idx in zip(axes, range(num_images)):
        image_tensor, annotations = dataset[idx]

        # (C, H, W) → (H, W, C)
        image = image_tensor.permute(1, 2, 0).numpy().clip(0, 1)
        ax.imshow(image)
        ax.axis("off")

        # Get image filename as title
        img_id   = dataset.ids[idx]
        img_info = coco.loadImgs(img_id)[0]
        ax.set_title(img_info["file_name"].split("/")[-1], fontsize=8)

        for ann in annotations:
            if ann.get("iscrowd", 0):       # skip crowd annotations
                continue

            cat_id = ann["category_id"]
            label  = id2label.get(cat_id, str(cat_id))
            color  = COLORS[cat_id]

            # COCO bbox: [x_min, y_min, width, height]
            x, y, w, h = ann["bbox"]

            rect = patches.Rectangle(
                (x, y), w, h,
                linewidth=2,
                edgecolor=color,
                facecolor="none",
            )
            ax.add_patch(rect)

            # Label background box
            ax.text(
                x, y - 4,
                label,
                fontsize=7,
                color="white",
                bbox=dict(facecolor=color, edgecolor="none", pad=1.5, alpha=0.85),
            )

    # Hide unused axes
    for ax in axes[num_images:]:
        ax.axis("off")

    plt.suptitle("COCO Detection — val2017", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
    print("✅ Saved → coco_detection_preview.png")

visualize_coco(dataset, num_images=6, cols=3)

## 2.5 Datasets publicos

### 2.5.1 [ImageNet](https://ieeexplore.ieee.org/document/5206848/): A large-scale hierarchical image database
he ImageNet dataset contains 14,197,122 images annotated according to the WordNet hierarchy. Since 2010, the dataset has been used in the ImageNet Large Scale Visual Recognition Challenge (ILSVRC), a benchmark for image classification and object detection.

The published dataset includes a training set of manually annotated images. A test image set is also provided, but its manual annotations are kept hidden.


### 2.5.2 [Microsoft COCO](https://arxiv.org/pdf/1405.0312v3): Common Objects in Context
The **MS COCO (Microsoft Common Objects in Context)** dataset is a large-scale dataset designed for object detection, segmentation, keypoint detection, and image captioning. The dataset contains **328,000 images**.

**Annotations**

The dataset provides annotations for several tasks:

* **Object Detection**: Bounding boxes and instance segmentation masks for **80 object categories**.
* **Image Captioning**: Natural language descriptions of images (see **MS COCO Captions**).
* **Keypoint Detection**: Contains more than **200,000 images** and **250,000 labeled person instances** with keypoints (**17 possible keypoints**, such as left eye, nose, right hip, and right ankle).
* **Stuff Segmentation**: Pixel-level segmentation masks with **91 “stuff” categories**, such as grass, wall, and sky (see **MS COCO Stuff**).
* **Panoptic Segmentation**: Full scene segmentation combining **80 “thing” categories** (e.g., person, bicycle, elephant) and a subset of **91 “stuff” categories** (e.g., grass, sky, road).
* **DensePose**: More than **39,000 images** and **56,000 person instances** labeled with DensePose annotations. Each labeled person includes an instance ID and a mapping between the image pixels belonging to that person’s body and a **template 3D model**.



### 2.5.3 [CelebA (CelebFaces Attributes Dataset)](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html)
The **CelebFaces Attributes (CelebA)** dataset contains **202,599 face images** of size **178 × 218** from **10,177 celebrities**, each annotated with **40 binary labels** indicating facial attributes such as hair color, gender, and age.


### 2.5.4 [KITTI](http://www.cvlibs.net/datasets/kitti/)
KITTI (Karlsruhe Institute of Technology and Toyota Technological Institute) is one of the most widely used datasets for mobile robotics and autonomous driving. It consists of hours of traffic scenarios recorded using multiple sensor modalities, including high-resolution RGB cameras, grayscale stereo cameras, and a 3D laser scanner.

### 2.5.5 [nuScenes](https://www.nuscenes.org/)
The nuScenes dataset is a large-scale dataset for autonomous driving. It includes 3D bounding box annotations for 1,000 scenes collected in Boston and Singapore. Each scene lasts 20 seconds and is annotated at 2 Hz.

# 3 - Convolutional Neural Networks (CNN)
A CNN is a neural network designed to process grid-structured data (images) by learning spatial hierarchies of features through convolutional filters.The core idea: instead of connecting every pixel to every neuron (expensive, ignores spatial structure), a small filter slides across the image learning local patterns. **Convolutional Neural Networks (CNNs)** typically follow a modular structure where the architecture can be understood as having **three distinct roles** :
- Backbone / Encoder
- Feature Aggregation / Representation Learning (Neck)
- Task-Specific Prediction (Head / Decoder)


## 3.1 CNN structure



### 3.1.1. Feature Extraction (Backbone / Encoder)

The first part of the CNN extracts **low-level and high-level features** from the input image using convolutional layers.

* Early layers detect **basic patterns** such as edges, corners, and textures.
* Deeper layers capture **more complex patterns**, such as object parts or shapes.
* Common components:

  * Convolution layers
  * Activation functions (ReLU, GELU)
  * Pooling layers
  * Batch normalization

Examples of backbones: **ResNet, VGG, EfficientNet, MobileNet**.



### 3.1.2. Feature Aggregation / Representation Learning (Neck)

The second part processes and refines the extracted features to produce a **useful representation** for the final task.

Typical operations:

* Combining multi-scale features
* Enhancing spatial or semantic information
* Reducing dimensionality

Examples:

* **Feature Pyramid Networks (FPN)**
* **BiFPN**
* Attention modules

This stage is common in tasks like **object detection and segmentation**.




### 3.1.3 Task-Specific Prediction (Head / Decoder)

The final component converts the learned representation into **task outputs**.

The structure depends on the task:

* **Classification** → Fully connected layers + Softmax
* **Object Detection** → Bounding box regression + class prediction
* **Segmentation** → Pixel-wise classification maps
* **Keypoint detection** → Heatmaps for landmarks

Examples:

* YOLO detection head
* Mask R-CNN segmentation head
* U-Net decoder

### Summary

| Role         | Purpose                       | Typical Components    |
| ------------ | ----------------------------- | --------------------- |
| **Backbone** | Extract visual features       | Conv layers, pooling  |
| **Neck**     | Aggregate and refine features | FPN, attention        |
| **Head**     | Produce final predictions     | Classifiers, decoders |

This **backbone–neck–head paradigm** is widely used in modern vision systems such as **YOLO, Faster R-CNN, Mask R-CNN, and many transformer-based vision models**.

In [ ]:
display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/convolutions.gif'))


## 3.2 Setup and imports

We need:
- **torch** and **torch.nn**: define and run the CNN.
- **torchvision**: load a sample image and the pre-trained ResNet50 model.
- **matplotlib**: plot the original image and the feature maps (activations) after each layer.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from torchvision import transforms
from torchvision.io import read_image
from pathlib import Path

# Use CPU for simplicity (change to "cuda" if you have a GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3.3 Custom Convolutional Neural Network

We define a **small CNN** with several blocks. Each block contains:
- **Conv2d**: learns local patterns (edges, textures) via kernels. Output has multiple *channels* (feature maps).
- **ReLU**: non-linearity so the network can approximate complex functions.
- **MaxPool2d**: downsamples the spatial dimensions, making the representation more compact and invariant to small shifts.

The **output size** after each layer can be computed from:  
`out_size = (in_size + 2*padding - kernel_size) / stride + 1`  
(For max pool, same formula with kernel and stride of the pool.)

In [ ]:
class CustomCNN(nn.Module):
    """Simple CNN: Conv -> ReLU -> Pool repeated, then flatten and classify."""

    def __init__(self):
        super().__init__()
        # Block 1: 3 -> 16 channels, spatial size reduced by pool
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)   # (1, 3, 64, 64) -> (1, 16, 64, 64)
        self.pool1 = nn.MaxPool2d(2, 2)                            # -> (1, 16, 32, 32)

        # Block 2: 16 -> 32 channels
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # -> (1, 32, 32, 32)
        self.pool2 = nn.MaxPool2d(2, 2)                            # -> (1, 32, 16, 16)

        # Block 3: 32 -> 64 channels
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # -> (1, 64, 16, 16)
        self.pool3 = nn.MaxPool2d(2, 2)                            # -> (1, 64, 8, 8)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = self.pool3(self.relu(self.conv3(x)))
        return x

model = CustomCNN().to(device)
model.eval()

### 3.3.1 The activation function
This activation function is responsible for determining the final output of each convolution computation and introduce non-linearities in our network, allowing it to model non-linear data. This way, it can stack convolutions and introduce the concept of “depth”, since stacking linear transformations is the same as having only one linear transformation. Thus, introducing this non-linearity is essential for our deep neural networks. The most popular activation function is called the ReLU function, which stands for Rectified Linear Unit. It is used right after a convolution inside what we call a “convolution unit”, or “conv”, as shown in the image below. It puts to zero any negative result making the convolution’s output more sparse, meaning that we have many zeros and a few important parameters. Thus “forcing” the network to focus on these parameters and be much more efficient to train in computation time, since a multiplication with zero, will always equal zero. It also helps to overcome the vanishing gradient problem, allowing models to learn faster and perform better, as we will discuss later.


In [ ]:
from IPython.display import Image, display
display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/relu.gif'))
display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/convolutions.gif'))


## 3.4 Capturing activations with hooks

To **visualize how the image is transformed after each layer**, we register **forward hooks** on the model. Each hook is a function that PyTorch calls after a layer runs, so we can store that layer’s output (the *activations* or *feature maps*). We store outputs after: Conv1+ReLU, Pool1, Conv2+ReLU, Pool2, Conv3+ReLU, Pool3.

In [ ]:
activations = {}

def get_activation(name):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

# Register hooks on the layers we want to visualize (after each block: conv+relu then pool)
model.conv1.register_forward_hook(get_activation("after_conv1_relu"))
model.pool1.register_forward_hook(get_activation("after_pool1"))
model.conv2.register_forward_hook(get_activation("after_conv2_relu"))
model.pool2.register_forward_hook(get_activation("after_pool2"))
model.conv3.register_forward_hook(get_activation("after_conv3_relu"))
model.pool3.register_forward_hook(get_activation("after_pool3"))


## 3.5 Load a sample image (COCO dataset)

We load one image from **COCO** (Common Objects in Context). If you have COCO at `./data/coco/val2017` and `./data/coco/annotations/instances_val2017.json`, that is used; otherwise one sample image is downloaded. Resized to 64×64 for the custom CNN. Tensor shape: **(batch, channels, height, width)** = (1, 3, H, W).

In [ ]:
from pathlib import Path
from PIL import Image
import urllib.request

coco_root = Path("./data/coco/val2017")
coco_ann = Path("./data/coco/annotations/instances_val2017.json")
img_size = 64  # resize to 64×64 for the custom CNN


if coco_root.exists() and coco_ann.exists():
    from torchvision.datasets import CocoDetection
    coco = CocoDetection(root=str(coco_root), annFile=str(coco_ann))
    img_pil, _ = coco[1]
    print("Loaded image from local COCO dataset.")
else:
    Path("./data").mkdir(exist_ok=True)
    url = "https://images.cocodataset.org/val2017/000000000139.jpg"
    sample_path = "./data/coco_sample.jpg"
    try:
        urllib.request.urlretrieve(url, sample_path)
    except urllib.error.URLError as e:
        # Ignore the next conditional block. There is a temporary SSL issue with coco dataset.
        # These line bypass ssl issues.
        print("[ WARN ] Coco SSL exception triggered ! By passing ssl verification !")
        if "SSL" in str(e) or "certificate" in str(e).lower():
            import ssl
            ssl_ctx = ssl.create_default_context()
            ssl_ctx.check_hostname = False
            ssl_ctx.verify_mode = ssl.CERT_NONE
            with urllib.request.urlopen(url, context=ssl_ctx) as r:
                with open(sample_path, "wb") as f:
                    f.write(r.read())
        else:
            raise
    img_pil = Image.open(sample_path).convert("RGB")
    print("Downloaded one COCO sample image (no local COCO found).")

## 3.6 Normalize dataset
### What is Normalization?
Normalization rescales pixel values from their original range [0, 255] (after ToTensor() converts to [0.0, 1.0]) to a centered distribution around zero.

The formula applied per channel is:

output = (pixel - mean) / std <br>
So for a red pixel value of 0.5: <br>
**(0.5 - 0.485) / 0.229 ≈ 0.065**

### Why Normalize?
1. Faster convergence — Gradient descent works better when inputs are on a similar scale. Without it, one channel dominating others causes unstable updates.
2. Zero-centered inputs — Values roughly in [-1, 1] avoid saturation in activation functions like sigmoid/tanh.
3. Matches pretrained weights — These specific values:
```
mean=[0.485, 0.456, 0.406]  # R, G, B
std= [0.229, 0.224, 0.225]
```

are the **ImageNet dataset statistics** — the mean and std of all ImageNet pixels. <br>
**If you're using a pretrained model (ResNet, VGG, ViT, etc.), the weights were trained expecting this exact distribution. Using different normalization = garbage output.**

### What Does std Do Specifically?
Standard deviation controls the spread/scale. Dividing by std makes the variance ≈ 1.
| Without std division  |  With std division |
|---|---|
| Values spread unevenly across channels  |  All channels have ~equal variance |
| Channels with high variance dominate gradients  | Balanced gradient flow across R, G, B  |

Think of it as "how many standard deviations away from the mean is this pixel?" — that's a statistically natural representation for a neural network to work with.

### Quick Summary
- StepRangePurposeRaw pixel[0, 255] <br>
- OriginalAfter ToTensor()[0.0, 1.0] <br>
- Scale downAfter Normalize()≈ [-2.1, 2.6] <br>
- Match ImageNet distribution<br>
<br></br>
**If you're training from scratch on your own dataset, you should compute your dataset's own mean/std instead of using ImageNet values.**


In [ ]:
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_just_normalization = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

x_norm = transform_just_normalization(img_pil).unsqueeze(0).to(device)
x = transform(img_pil).unsqueeze(0).to(device)  # (1, 3, 64, 64)

### 3.5.1 Display images


In [ ]:
# Show original image (denormalize for display) at 64×64
def denorm(t):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(t.device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(t.device)
    return t * std + mean

plt.figure(figsize=(3.2,3.2))
plt.axis("off")
plt.title("Original image")
plt.imshow(img_pil) # Original img

plt.figure(figsize=(3.2,3.2))
plt.imshow(x_norm.squeeze(0).permute(1, 2, 0).cpu().numpy().clip(0, 1)) # Normalized img
plt.axis("off")
plt.title("Normalized image")

plt.figure(figsize=(3.2, 3.2))
plt.imshow(denorm(x).squeeze(0).permute(1, 2, 0).cpu().numpy().clip(0, 1))
plt.axis("off")
plt.title("Input image (64×64, COCO)")
plt.show()
print(f"Input shape: {x.shape}  (batch, channels, height, width)")

## 3.6 Running CNN

In [ ]:
_ = model(x)

## 3.7 Visualizing the feature maps after each layer

Each **channel** of the activation tensor is a 2D *feature map*: the response of one filter over the spatial dimensions. Early layers often capture edges and simple textures; deeper layers capture more abstract patterns. We plot a grid of channels for each captured layer (only a subset of channels if there are many).

In [ ]:
def show_feature_maps(activations_dict, max_channels=16, figsize_per_block=(12, 3)):
    """Plot feature maps for each stored layer. Each row = one layer, each column = one channel."""
    layer_names = [
        "after_conv1_relu", "after_pool1",
        "after_conv2_relu", "after_pool2",
        "after_conv3_relu", "after_pool3",
    ]
    for name in layer_names:
        if name not in activations_dict:
            continue
        feat = activations_dict[name]  # (1, C, H, W)
        C, H, W = feat.shape[1], feat.shape[2], feat.shape[3]
        n_show = min(max_channels, C)
        fig, axes = plt.subplots(1, n_show, figsize=figsize_per_block)
        if n_show == 1:
            axes = [axes]
        for i in range(n_show):
            im = feat[0, i].cpu().numpy()
            axes[i].imshow(im, cmap="viridis")
            axes[i].set_title(f"ch {i}")
            axes[i].axis("off")
        plt.suptitle(f"{name}  shape: (1, {C}, {H}, {W})")
        plt.tight_layout()
        plt.show()

show_feature_maps(activations)

## 3.8 Custom Backbone + Neck + Head


### 3.8.1 Defining the new Backbone

In [ ]:
class CustomCNN(nn.Module):
    """Backbone that returns intermediate feature maps for multi-scale neck."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Return ALL stages — not just the last one
        c1 = self.pool1(self.relu(self.conv1(x)))   # (B, 16, 32, 32)  ← small objects
        c2 = self.pool2(self.relu(self.conv2(c1)))  # (B, 32, 16, 16)  ← medium objects
        c3 = self.pool3(self.relu(self.conv3(c2)))  # (B, 64,  8,  8)  ← large objects / semantics

        return c1, c2, c3   # multi-scale outputs

### 3.8.2 Defining the Custom Neck

In [ ]:
class FPNNeck(nn.Module):
    """
    Feature Pyramid Network neck.
    Fuses multi-scale backbone outputs top-down:
      c3 (deepest, smallest) → upsample → add with c2 → upsample → add with c1
    All outputs share the same channel count (out_channels).
    """

    def __init__(self, out_channels=64):
        super().__init__()

        # 1×1 lateral convs — align all backbone channels to out_channels
        self.lateral_c1 = nn.Conv2d(16, out_channels, kernel_size=1)  # 16 → 64
        self.lateral_c2 = nn.Conv2d(32, out_channels, kernel_size=1)  # 32 → 64
        self.lateral_c3 = nn.Conv2d(64, out_channels, kernel_size=1)  # 64 → 64

        # 3×3 output convs — smooth after adding upsampled features
        self.smooth_p1 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth_p2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth_p3 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.relu     = nn.ReLU(inplace=True)

    def forward(self, c1, c2, c3):
        # Align channels with lateral convs
        l1 = self.lateral_c1(c1)   # (B, 64, 32, 32)
        l2 = self.lateral_c2(c2)   # (B, 64, 16, 16)
        l3 = self.lateral_c3(c3)   # (B, 64,  8,  8)

        # Top-down fusion — start from deepest
        p3 = self.relu(self.smooth_p3(l3))                          # (B, 64,  8,  8)
        p2 = self.relu(self.smooth_p2(l2 + self.upsample(p3)))      # (B, 64, 16, 16)
        p1 = self.relu(self.smooth_p1(l1 + self.upsample(p2)))      # (B, 64, 32, 32)

        return p1, p2, p3   # multi-scale fused features

### 3.8.3 Defining the classification head

In [ ]:
class ClassificationHead(nn.Module):
    """
    Takes the deepest pyramid level (p3) and classifies the whole image.
    p3: (B, 64, 8, 8) → num_classes logits
    """

    def __init__(self, in_channels=64, num_classes=10):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d((1, 1))   # (B, 64, 8, 8) → (B, 64, 1, 1)
        self.flat = nn.Flatten()                   # → (B, 64)

        self.head = nn.Sequential(
            nn.Linear(in_channels, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),           # → (B, num_classes)
        )

    def forward(self, p3):
        x = self.pool(p3)
        x = self.flat(x)
        return self.head(x)                        # raw logits

### 3.8.5 Defining the detection head

In [ ]:
class DetectionHead(nn.Module):
    """
    Runs on each FPN level independently.
    Per spatial location predicts:
      - num_anchors × 4      → bbox offsets  (dx, dy, dw, dh)
      - num_anchors × num_classes → class scores
    """

    def __init__(self, in_channels=64, num_anchors=3, num_classes=10):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

        # Two separate output branches
        self.cls_branch = nn.Conv2d(in_channels, num_anchors * num_classes, kernel_size=1)
        self.reg_branch = nn.Conv2d(in_channels, num_anchors * 4,           kernel_size=1)

    def forward(self, features):
        """
        features: list of [p1, p2, p3] from neck
        returns:  list of (cls_output, reg_output) per scale
        """
        outputs = []
        for p in features:
            x   = self.shared(p)
            cls = self.cls_branch(x)   # (B, num_anchors×num_classes, H, W)
            reg = self.reg_branch(x)   # (B, num_anchors×4,           H, W)
            outputs.append((cls, reg))
        return outputs

### 3.8.6 Assemble the full detector

In [ ]:
class FullDetector(nn.Module):
    """
    Complete model:
      Image → Backbone → Neck → Head
    Supports both classification and detection tasks simultaneously.
    """

    def __init__(self, num_classes=10, num_anchors=3):
        super().__init__()

        self.backbone = CustomCNN()
        self.neck     = FPNNeck(out_channels=64)
        self.cls_head = ClassificationHead(in_channels=64, num_classes=num_classes)
        self.det_head = DetectionHead(in_channels=64, num_anchors=num_anchors,
                                      num_classes=num_classes)

    def forward(self, x):
        # ── Backbone ──────────────────────────────────────
        c1, c2, c3 = self.backbone(x)           # multi-scale raw features

        # ── Neck ──────────────────────────────────────────
        p1, p2, p3 = self.neck(c1, c2, c3)      # fused pyramid features

        # ── Heads ─────────────────────────────────────────
        cls_logits  = self.cls_head(p3)          # (B, num_classes)
        det_outputs = self.det_head([p1, p2, p3])# list of (cls, reg) per scale

        return {
            "cls_logits":  cls_logits,   # image-level classification
            "det_outputs": det_outputs,  # per-scale detection outputs
        }

### 3.8.4 Using the detector

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FullDetector(num_classes=10, num_anchors=3).to(device)
model.eval()

# Dummy input — batch of 2 RGB images 64×64
x = torch.randn(2, 3, 64, 64).to(device)

with torch.no_grad():
    out = model(x)

# ── Classification output ──────────────────────────────────
print("cls_logits :", out["cls_logits"].shape)
# → (2, 10)

# ── Detection outputs per FPN level ───────────────────────
for i, (cls, reg) in enumerate(out["det_outputs"]):
    print(f"P{i+1}  cls: {cls.shape}   reg: {reg.shape}")

# 4- ResNet50: residual networks and why they work
## [Deep Residual Learning for Image Recognition Paper](https://arxiv.org/abs/1512.03385)

## 4.1 The vanishing gradient problem

In **very deep** networks (many layers), gradients can become very small as they are backpropagated. As a result, early layers learn slowly or almost not at all. This limits how deep we can train networks effectively.



### 4.2 Skip connections (residual connections)

**ResNet** (He et al., 2015) introduces **skip connections**: the input to a block is added to the output of the block before passing to the next layer:

$$y = F(x) + x$$

- **x**: input to the block  
- **F(x)**: output of the convolutional layers inside the block  
- **y**: we learn the *residual* (the change) instead of the full mapping

This way, the gradient can flow directly through the identity path ($x$), so it does not vanish. The network can learn to make $F(x)$ small when no change is needed, effectively using more or fewer layers as needed.



### 4.3 ResNet50 architecture in short

- **50** refers to roughly 50 *weight layers* (conv + fc ( fully connected)).
- **Stem**: one conv + max pool that reduce spatial size and increase channels.
- **Stages**: four stages, each made of several **bottleneck blocks**.
- **Bottleneck block**: 1×1 conv (reduce channels) → 3×3 conv → 1×1 conv (restore channels), plus a skip connection. The 1×1 convolutions make the block cheaper in parameters and compute.
- **Global average pool** then a **fully connected** layer for classification.

So ResNet50 is a deep CNN that stays trainable thanks to residual (skip) connections and uses bottleneck blocks for efficiency.

## 4.4 Bottleneck block (code)

A **bottleneck block** in ResNet looks like this:  
`1×1 conv (reduce C) → 3×3 conv → 1×1 conv (restore C)` and then **add the input** (skip). The 1×1 convolutions reduce and then restore the number of channels, so the 3×3 conv works on fewer channels and the block is cheaper.

In [ ]:
from IPython.display import display, Image
display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/res.png'))
display(Image(filename='media/pictures/Clase_1_Convoluciones_Datasets_y_CNN/resnet50.png'))

In [ ]:
class BottleneckBlock(nn.Module):
    """One ResNet bottleneck: 1x1 -> 3x3 -> 1x1 conv + skip connection."""

    def __init__(self, in_ch, mid_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, mid_ch, 1, stride=1, bias=False)
        self.conv2 = nn.Conv2d(mid_ch, mid_ch, 3, stride=stride, padding=1, bias=False)
        self.conv3 = nn.Conv2d(mid_ch, out_ch, 1, bias=False)
        self.relu = nn.ReLU(inplace=True)
        # If spatial size or channels change, we need to project the skip to match
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)
            )

    def forward(self, x):
        identity = x
        out = self.relu(self.conv1(x))
        out = self.relu(self.conv2(out))
        out = self.conv3(out)
        out = out + self.shortcut(identity)  # skip connection
        out = self.relu(out)
        return out

# Example: 256 -> 64 -> 64 -> 256 channels, same spatial size
block = BottleneckBlock(256, 64, 256)
dummy = torch.randn(1, 256, 32, 32)
print("Bottleneck block output shape:", block(dummy).shape)

## 4.5 Using pre-trained ResNet50 with ImageNet Dataset

We load **ResNet50** from `torchvision.models` (pre-trained on ImageNet) and run a forward pass. The model expects input of size **224×224**. We use the same normalization (ImageNet mean/std) as before.

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

# Load pre-trained ResNet50
weights = ResNet50_Weights.IMAGENET1K_V1
categories = weights.meta["categories"]  # list of 1000 class names

resnet = resnet50(weights=weights).to(device)
resnet.eval()

# ResNet expects 224x224; resize our image
transform_224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
x_224 = transform_224(img_pil).unsqueeze(0).to(device)

with torch.no_grad():
    logits = resnet(x_224)

probs = torch.softmax(logits, dim=1)
top5_prob, top5_idx = torch.topk(probs[0], 5)
print("ResNet50 top-5 predictions (ImageNet):")
for i in range(5):
    idx = top5_idx[i].item()
    print(f"  {i+1}. class index {idx} [{categories[idx]}]  prob = {top5_prob[i].item():.4f}")

## 4.6 ResNet50 layer overview

ResNet50 is built from a **stem** (conv + bn + relu + maxpool) and **4 stages** of bottleneck blocks. The following lists the main submodules so you can see how the 50 layers are organized.

In [ ]:
# Inspect ResNet50 structure
for name, child in resnet.named_children():
    n_params = sum(p.numel() for p in child.parameters())
    print(f"{name:12s}  parameters: {n_params:>10,}")

## 4.7 Visualizing pre-trained kernels vs custom kernels

In [ ]:
def show_without_filters(modelo, titulo, ax_grid):
    # 1. Acceder a los filtros de conv1: shape [64, 3, 7, 7]
    filtros = modelo.conv1.weight.data.clone().cpu()  # [64, 3, 7, 7]

    for i, ax in enumerate(ax_grid.flat):
        f = filtros[i]  # [3, 7, 7] — un filtro RGB

        # 2. Normalización min-max para visualizar como imagen
        f_norm = (f - f.min()) / (f.max() - f.min() + 1e-8)  # [3, 7, 7] en [0, 1]

        # Reordenar a [H, W, C] para imshow
        ax.imshow(f_norm.permute(1, 2, 0))
        ax.axis("off")

    ax_grid.flat[0].set_title(titulo, fontsize=10, pad=4)

fig, axes = plt.subplots(3, 8, figsize=(16, 5))
#                        ^  ^
#                2 modelos  8×8 filtros c/u

resnet_no_train = resnet50().to(device)
resnet_no_train.eval()
show_without_filters(model.backbone, "Custom Model - (sin entrenar): 64 filtros de 3×3×3", axes[0])
show_without_filters(resnet_no_train, "ResNet50 - No weights: 64 filtros de 3×7×7", axes[1])
show_without_filters(resnet, "ImageNet (pre-entrenado) - ResNet50: 64 filtros de 3×7×7", axes[2])

plt.suptitle("Filtros conv1 ", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# 5 - FastRCNN and CocoDataset with BBoxes
 We are going to re-use dataset loaded in step *3.5* but redifining transformation function.

In [ ]:
from PIL import Image

dataset = CocoDetection(
    root=coco_root,
    annFile=coco_ann,
    transform=transforms.ToTensor()
)


## 5.1 Loading pre-trained model FasterRCNN

In [ ]:

from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
categories = weights.meta["categories"]

model   = fasterrcnn_resnet50_fpn(weights=weights)
model.eval()

img_tensor, annotations = dataset[0]
ID_TO_NAME = {idx: cat for idx,cat in enumerate(categories)}
MODEL_LABELS=weights.meta["categories"]

object_1 = annotations[0]["category_id"]
object_1_name = ID_TO_NAME[object_1]
object_1_segment = annotations[0]["segmentation"]
object_1_bboox = annotations[0]["bbox"]

print("Image tensor shape:", img_tensor.shape)   # [3, H, W]
print("Number of objects annotated:", len(annotations))
print(f"Object 1 : {object_1} - {object_1_name}")
print(f"Segmentation mask object 1 : {object_1_segment}")
print(f"Bounding box object 1 : {object_1_bboox}")


### 5.2 Transform coco coordinates from x,y,w,h to xy,xy

In [ ]:
def coco_bbox_to_xyxy(bbox):
    """[x, y, w, h] → [x1, y1, x2, y2]"""
    x, y, w, h = bbox
    return [x, y, x + w, y + h]

### 5.3 Draw BBoxes

In [ ]:
from torchvision.utils import draw_bounding_boxes


In [ ]:
def draw_ground_truth(dataset, idx):
    img_tensor, annotations = dataset[idx]
    if len(annotations) == 0:
        print("No annotations for this image.")
        return

    boxes = torch.tensor(
        [coco_bbox_to_xyxy(ann["bbox"]) for ann in annotations],
        dtype=torch.float32
    )
    labels = [ID_TO_NAME.get(ann["category_id"], "?") for ann in annotations]

    img_uint8 = (img_tensor * 255).byte()

    drawn = draw_bounding_boxes(
        img_uint8,
        boxes=boxes,
        labels=labels,
        colors="green",
        width=2,
        font_size=12
    )

    plt.figure(figsize=(12, 8))
    plt.imshow(drawn.permute(1, 2, 0).numpy())
    plt.title(f"Ground Truth — idx {idx} | {len(annotations)} objects")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

draw_ground_truth(dataset, idx=0)

In [ ]:
def compare_gt_vs_pred(dataset, idx, threshold=0.5):
    img_tensor, annotations = dataset[idx]
    img_uint8 = (img_tensor * 255).byte()

    # ── Ground Truth ──────────────────────────────────────
    gt_boxes = torch.tensor(
        [coco_bbox_to_xyxy(a["bbox"]) for a in annotations],
        dtype=torch.float32
    ) if annotations else torch.zeros((0, 4))

    gt_labels = [ID_TO_NAME.get(a["category_id"], "?") for a in annotations]

    # ── Predictions ───────────────────────────────────────
    with torch.no_grad():
        preds = model([img_tensor])[0]

    mask   = preds["scores"] > threshold
    p_boxes  = preds["boxes"][mask]
    p_labels = preds["labels"][mask]
    p_scores = preds["scores"][mask]

    pred_label_strs = [
        f"{MODEL_LABELS[l]} {s:.2f}"
        for l, s in zip(p_labels.tolist(), p_scores.tolist())
    ]

    # ── Draw ──────────────────────────────────────────────
    drawn_gt = draw_bounding_boxes(
        img_uint8.clone(), boxes=gt_boxes,
        labels=gt_labels, colors="green", width=2, font_size=11
    ) if len(gt_boxes) > 0 else img_uint8.clone()

    drawn_pred = draw_bounding_boxes(
        img_uint8.clone(), boxes=p_boxes,
        labels=pred_label_strs, colors="red", width=2, font_size=11
    ) if len(p_boxes) > 0 else img_uint8.clone()

    # ── Plot ──────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    axes[0].imshow(drawn_gt.permute(1, 2, 0).numpy())
    axes[0].set_title(f"Ground Truth ({len(gt_boxes)} objects)", fontsize=13)
    axes[0].axis("off")

    axes[1].imshow(drawn_pred.permute(1, 2, 0).numpy())
    axes[1].set_title(f"Predictions @ {threshold} ({len(p_boxes)} objects)", fontsize=13)
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

compare_gt_vs_pred(dataset, idx=2, threshold=0.2)

In [ ]:
for idx in range(5):
    compare_gt_vs_pred(dataset, idx=idx, threshold=0.2)